# 04 MLP across F0, F1, F2

Step 10. Training runs from `python -m src.train_mlp` (model in `src/models/mlp.py`), which appends every CV fold and validation seed to `results/log.csv`. This notebook only reads the log, so re-running it never retrains.

Input: `LinearPipeline` output (provisional V-column list), refit on each training part. Two ReLU hidden layers (hidden, hidden/2) with dropout, Adam, batch 2,048, `BCEWithLogitsLoss(pos_weight = neg/pos)` from training rows, early stopping on validation PR-AUC with patience 5 (best epoch restored). Grid: hidden {128, 256} x dropout {0.1, 0.3} x learning rate {3e-4, 1e-3}, tuned with time CV (seed 0); the best config per feature set is refit on the train block and scored on validation for seeds 0, 1, 2.

In [ ]:
import sys; sys.path.insert(0, '..')
import json
import pandas as pd
import matplotlib.pyplot as plt
from src.results import LOG_PATH
from src.data import ROOT

log = pd.read_csv(LOG_PATH)
assert (log['split'] != 'test').all(), 'test block must stay unscored'
m = log[log.model == 'mlp'].copy()
params = m['params'].map(json.loads)
m['config'] = params.map(lambda p: f"h={p['hidden']} d={p['dropout']} lr={p['lr']:g}")
m['fold'] = params.map(lambda p: p.get('fold'))
m['fit'] = params.map(lambda p: p.get('_fit', {}))
m['best_epoch'] = m['fit'].map(lambda f: f.get('best_epoch'))
m['epochs_run'] = m['fit'].map(lambda f: f.get('epochs_run'))
cv, val = m[m.split == 'cv'], m[m.split == 'val']
print(len(cv), 'cv rows,', len(val), 'validation rows')

## Time-CV PR-AUC by config (mean over 4 folds)

In [ ]:
pd.set_option('display.precision', 4)
cv.pivot_table(index='config', columns='feature_set', values='pr_auc', aggfunc='mean')

In [ ]:
cv.groupby('feature_set')[['best_epoch', 'epochs_run']].describe().round(1)

## Validation block: selected config, seeds 0, 1, 2

In [ ]:
val[['feature_set', 'seed', 'config', 'pr_auc', 'auroc', 'precision', 'recall',
     'recall_top1pct', 'best_epoch', 'epochs_run', 'runtime_s']].sort_values(['feature_set', 'seed'])

In [ ]:
val.groupby('feature_set').agg(pr_auc_mean=('pr_auc', 'mean'), pr_auc_std=('pr_auc', 'std'),
                               auroc_mean=('auroc', 'mean'), recall_top1pct_mean=('recall_top1pct', 'mean'))

## Training curves (validation fits)

Training loss (weighted BCE, mean over the epoch) and validation PR-AUC by epoch for each feature set and seed. The marker shows the restored best epoch.

In [ ]:
fsets = sorted(val.feature_set.unique())
fig, axes = plt.subplots(2, len(fsets), figsize=(4.2 * len(fsets), 6), sharex=True, squeeze=False)
for j, fs in enumerate(fsets):
    for _, r in val[val.feature_set == fs].sort_values('seed').iterrows():
        f = r['fit']; ep = range(1, f['epochs_run'] + 1)
        line, = axes[0, j].plot(ep, f['loss'], marker='.', label=f"seed {r['seed']}")
        axes[1, j].plot(ep, f['val_pr_auc'], marker='.', color=line.get_color())
        b = f['best_epoch']
        axes[1, j].scatter([b], [f['val_pr_auc'][b - 1]], s=60, facecolors='none', edgecolors=line.get_color())
    axes[0, j].set_title(f'{fs}: {val[val.feature_set == fs].config.iloc[0]}', fontsize=9)
    axes[1, j].set_xlabel('epoch')
axes[0, 0].set_ylabel('training loss'); axes[1, 0].set_ylabel('validation PR-AUC')
axes[0, 0].legend(fontsize=8)
fig.suptitle('MLP training curves (validation fits)'); fig.tight_layout()
fig.savefig(ROOT / 'reports' / 'figures' / 'mlp_training_curves.png', dpi=150)